In [1]:
import pandas as pd
import numpy as np

# ______________________LOAD PREPROCESSED DATA ___________________________________

df = pd.read_csv("admission_cohort.csv")

print("Original Shape:", df.shape)


Original Shape: (546028, 28)


In [2]:

# __________________________DATETIME FEATURES _______________________________-


df["admittime"] = pd.to_datetime(df["admittime"])
df["dischtime"] = pd.to_datetime(df["dischtime"])


In [3]:

# __________________________PRIOR ADMISSIONS ____________________________________


df = df.sort_values(
    ["subject_id", "admittime"]
)

df["prior_admissions"] = (
    df.groupby("subject_id")
    .cumcount()
)


# ___________________________ADMISSION MONTH _______________________________________-


df["admission_month"] = (
    df["admittime"]
    .dt.month)


# _________________________________DAY OF WEEK _____________________________________

df["admission_dayofweek"] = (
    df["admittime"]
    .dt.dayofweek
)


# ___________________________ WEEKEND ADMISSION ____________________________


df["weekend_admission"] = np.where(
    df["admission_dayofweek"] >= 5,
    1,
    0)


# ________________________ADMISSION HOUR __________________________________

df["admission_hour"] = (
    df["admittime"]
    .dt.hour)

In [4]:

#_____________________________- HIGH RISK AGE FLAG __________________________________

df["elderly_patient"] = np.where(
    df["age"] >= 65,
    1,
    0)


In [5]:

# ____________________________LONG STAY FLAG _________________________________


df["long_stay"] = np.where(
    df["length_of_stay_days"] > 7,
    1,
    0
)

In [6]:

# _____________________________COMPLEX CARE FLAG __________________________

df["complex_case"] = np.where(
    (
        df["transfer_count"] >= 3
    ) |
    (
        df["service_count"] >= 3
    ),
    1,
    0)


# _______________________________HIGH SEVERITY FLAG ______________________________

df["high_severity"] = np.where(
    df["avg_drg_severity"] >= 3,
    1,
    0
)

# _______________________________HIGH MORTALITY FLAG __________________________

df["high_mortality_risk"] = np.where(
    df["avg_drg_mortality"] >= 3,
    1,
    0)

In [7]:

# ____________________________HANDLE MISSING VALUES _______________________________

numeric_columns = df.select_dtypes(
    include=np.number
).columns

for col in numeric_columns:
    df[col] = df[col].fillna(
        df[col].median()
    )

categorical_columns = [
    "gender",
    "race",
    "insurance",
    "marital_status",
    "admission_type",
    "admission_location",
    "primary_service"
]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].fillna(
            "Unknown"
        )

In [8]:

# _________________________________SAVE OUTPUT ____________________________

df.to_csv(
    "model_dataset.csv",
    index=False)

print("\nFeature Engineering Complete")

print(
    "Output File:",
    "model_dataset.csv")

print(
    "Final Shape:",
    df.shape)

print(
    "\nNew Features Created:")

new_features = [
    "prior_admissions",
    "admission_month",
    "admission_dayofweek",
    "weekend_admission",
    "admission_hour",
    "elderly_patient",
    "long_stay",
    "complex_case",
    "high_severity",
    "high_mortality_risk"
]

for feature in new_features:
    print(feature)


Feature Engineering Complete
Output File: model_dataset.csv
Final Shape: (546028, 38)

New Features Created:
prior_admissions
admission_month
admission_dayofweek
weekend_admission
admission_hour
elderly_patient
long_stay
complex_case
high_severity
high_mortality_risk
